In [ ]:
#read the parquet file
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
import math
from datetime import date
from functions import *
from scipy.interpolate import PchipInterpolator


Might be to expensive to hedge vega
Consider special cases that we know will happen, we can obviously not hedge directly but we can look at markers that signals that the event will occur and hedge against that.

**Quantitative strategies**




In [ ]:
df = pd.read_parquet("unpivoted_bid_offer_train_clean.parquet", engine="pyarrow")

required = [
        "option_bid", "option_ask",
        "underlying_settlement", "strike",
        "expiration", "date", "option_type"
    ]

df = df.dropna(subset=required)
df['date'] = pd.to_datetime(df['date'])
df.head()

print("data range:", df['date'].min(), "to", df['date'].max())

In [ ]:

fixing_dates = pd.bdate_range(
    start="2021-02-01",
    end="2021-02-28"
)

print("len(fixing_dates):", len(fixing_dates))

valuation_date = pd.Timestamp("2020-01-06")

theta = (
    (fixing_dates - valuation_date).days / 365
).to_numpy()


In [ ]:
futures_df = (
    df[[
        "date",
        "future_contract_code",
        "underlying_settlement",
        "underlying_bid",
        "underlying_ask"
    ]]
    .drop_duplicates(subset=["date", "future_contract_code"])
    .copy()
)
options_df = df.dropna(
    subset=[
        "option_bid",
        "option_ask",
        "strike",
        "underlying_settlement",
        "expiration"
    ]
).copy()


options_df["mid_price"] = (
    options_df["option_bid"] + options_df["option_ask"]
) / 2


options_df["settlement_minus_mid"] = (
    options_df["settlement_price"] -
    options_df["mid_price"]
)

print(
    options_df[
        ["settlement_price", "mid_price",
         "settlement_minus_mid"]
    ].describe()
)

In [ ]:
print(futures_df["future_contract_code"].unique())
futures_df.head()

In [ ]:
N = fixing_dates.size
print("N:", N)

n1, n2 = N-1,1
w1, w2 = n1/N, n2/N



In [ ]:
daily_options = options_df[
    options_df["date"] == "2020-01-06"
].copy()

print(
    daily_options[
        ["future_contract_code", "option_contract_code"]
    ].drop_duplicates()
)

In [ ]:
valuation_date = pd.Timestamp("2020-01-06")

apr21_options = daily_options[
    daily_options["future_contract_code"] == "COJ21"
].copy()

may21_options = daily_options[
    daily_options["future_contract_code"] == "COK21"
].copy()

print("Apr21 observations:", len(apr21_options))
print("May21 observations:", len(may21_options))

In [ ]:
print(
    apr21_options["underlying_settlement"].unique()
)

print(
    may21_options["underlying_settlement"].unique()
)

F1 = 61.4
F2 = 61.05
Favg = w1*F1+w2*F2

In [ ]:
columns = [
    "strike",
    "option_type",
    "expiration",
    "time_to_expiry_days",
    "option_bid",
    "option_ask",
    "settlement_price"
]

print("APR21 OPTIONS")
print(apr21_options[columns].sort_values("strike").to_string(index=False))

print("\nMAY21 OPTIONS")
print(may21_options[columns].sort_values("strike").to_string(index=False))

In [ ]:


# =============================================================================
# PART 2: DATA LOADING AND DYNAMIC CORRELATION ESTIMATION
# =============================================================================

#create path variable for the parquet file
file_path = "unpivoted_bid_offer_train_clean.parquet"  # Adjust the path as necessary

# 1. Load the clean parquet dataset
df = pd.read_parquet(file_path)
df['date'] = pd.to_datetime(df['date'])

valuation_date = pd.Timestamp('2020-01-06')
d = df[df['date'] == valuation_date]

c = d[d['underlying_ticker'] == 'COJ21 Comdty']   # April 2021 futures contract
c_K = d[d['underlying_ticker'] == 'COK21 Comdty'] # May 2021 futures contract

f1 = c['underlying_settlement'].iloc[0]
f2 = c_K['underlying_settlement'].iloc[0]


# Compute rho as of the valuation date
rho = estimate_rho(df, valuation_date)
print("Estimated correlation rho:", rho)
# =============================================================================
# PART 3: SMILE CALIBRATION (Polynomial vs PCHIP Interpolation)
# =============================================================================

# --- April contract (COJ21) smile extraction ---
strikes_1, vols_1 = [], []
for _, ligne in c.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes_1.append(K)
        vols_1.append(vol_implicite(prix, F, K, T))

coef_poly_1 = np.polyfit(strikes_1, vols_1, 3) # Degree-3 polynomial (for visual comparison)
smile_apr21 = PchipInterpolator(strikes_1, vols_1, extrapolate=True) # Robust PCHIP interpolator


# --- May contract (COK21) smile extraction ---
strikes_2, vols_2 = [], []
for _, ligne in c_K.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes_2.append(K)
        vols_2.append(vol_implicite(prix, F, K, T))

#uses both methods to compare the implied volatility smiles for the May contract
coef_poly_2 = np.polyfit(strikes_2, vols_2, 3)
smile_may21 = PchipInterpolator(strikes_2, vols_2, extrapolate=True)



# =============================================================================
# OUTPUTS AND COMPARATIVE PLOT (Polynomial vs PCHIP)
# =============================================================================

# Comparative plot: Polynomial degree 3 vs PCHIP
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
x_grid = np.linspace(min(strikes_1), max(strikes_1), 100)
plt.plot(strikes_1, vols_1, 'ko', label='Market Implied Vols')
plt.plot(x_grid, np.polyval(coef_poly_1, x_grid), 'b--', label='Degree-3 Polynomial')
plt.plot(x_grid, smile_apr21(x_grid), 'r-', label='PCHIP Interpolation')
plt.title('Smile Comparison: COJ21 (April)')
plt.xlabel('Strike')
plt.ylabel('Implied Volatility')
plt.legend()

plt.subplot(1, 2, 2)
x_grid_K = np.linspace(min(strikes_2), max(strikes_2), 100)
plt.plot(strikes_2, vols_2, 'ko', label='Market Implied Vols')
plt.plot(x_grid_K, np.polyval(coef_poly_2, x_grid_K), 'b--', label='Degree-3 Polynomial')
plt.plot(x_grid_K, smile_may21(x_grid_K), 'r-', label='PCHIP Interpolation')
plt.title('Smile Comparison: COK21 (May)')
plt.xlabel('Strike')
plt.ylabel('Implied Volatility')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
valuation_date = pd.Timestamp("2020-01-06")
fixing_dates = pd.bdate_range("2021-02-01", "2021-02-28")

fixing_schedule = build_fixing_schedule(
    fixing_dates=fixing_dates,
    valuation_date=valuation_date,
    front_expiry="2021-02-26",
    front_contract="Apr21",
    next_contract="May21",
)

theta = fixing_schedule["theta"].to_numpy()
N = len(fixing_schedule)
n1 = int(fixing_schedule["contract"].eq("Apr21").sum())
n2 = N - n1
w1, w2 = n1 / N, n2 / N

display(fixing_schedule)

In [ ]:
market_data = pd.read_parquet("unpivoted_bid_offer_train_clean.parquet")
market_data["date"] = pd.to_datetime(market_data["date"])

training = market_data.loc[
    market_data["date"].between("2020-01-06", "2021-04-06")
]
target_quotes = training.loc[
    training["underlying_ticker"].isin(["COJ21 Comdty", "COK21 Comdty"])
]

futures_history = target_quotes.loc[
    np.isfinite(target_quotes["underlying_settlement"])
    & target_quotes["underlying_settlement"].gt(0),
    ["date", "underlying_ticker", "underlying_settlement"],
].drop_duplicates(["date", "underlying_ticker"])

daily_calibrations = pd.DataFrame.from_dict(
    {
        day: calibrate_day(quotes, futures_history, day)
        for day, quotes in training.groupby("date", sort=True)
    },
    orient="index",
).rename_axis("date")

calibration_summary = daily_calibrations.drop(
    columns=["smile_apr21", "smile_may21"]
)
display(calibration_summary)

In [ ]:
opening = daily_calibrations.loc[pd.Timestamp("2020-01-06")]

F1, F2 = opening["F_apr21"], opening["F_may21"]
Favg = w1 * F1 + w2 * F2
rho = opening["rho"]
smile_apr21 = opening["smile_apr21"]
smile_may21 = opening["smile_may21"]

In [ ]:
K_call = 78.0

sigma_apr, sigma_may = get_volatility(
    F1, F2, Favg, K_call, smile_apr21, smile_may21
)

call_price, _, call_se, _ = simulate_correlated_futures(
    F1=F1,
    F2=F2,
    sigma1=sigma_apr,
    sigma2=sigma_may,
    rho=rho,
    theta=theta,
    n1=n1,
    D=1.0,
    K=K_call,
    n_simulations=100_000,
)

print(f"Asian call: {call_price:.4f} USD/bbl (MC SE {call_se:.4f})")

In [ ]:
collar = solve_zero_cost_collar(
    F1=F1,
    F2=F2,
    smile_apr21=smile_apr21,
    smile_may21=smile_may21,
    rho=rho,
    theta=theta,
    n1=n1,
)

display(pd.Series(collar).to_frame("Value"))